In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import h5py
import anndata as ad

In [49]:
# read raw data
data_dir = '../data/frangieh'
adata = sc.read_h5ad(f"{data_dir}/rna.h5ad")

# preprocess
from preprocessing import preprocess

adata = preprocess(adata, 200, 200, 50_000, 15, plot=False) # values used in the original data: >200 genes, <18% mitochondrial genes, >200 cells

adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata, base=2) # use base=2 to log2 scale

In [50]:
# gene list for differentially expressed genes from the paper:
gene_list = [
    'KRT17',
    'TEX29',
    'CXCL9',
    'CD7',
    'CD74',
    'IDO8',
    'HLA-DRB1',
    'CXCL11',
    'HLA-DPA1',
    'PTGES',
    'HAPLN3',
    'TXNIP',
    'BATF3',
    'CD274',
    'HLA-DRA',
    'DDIT4',
    'CYP2S1',
    'SCN9A',
    'DOCK10',
    'HLA-B',
    'PRSS23',
    'RNF213',
    'GBP1',
    'GDF15',
    'SECTM1',
    'AC016831.1',
    'VSNL1',
    'GBP5',
    'HLA-DPB1',
    'PDCD1LG2',
]
adata.var['gene_list']=adata.var.index.isin(gene_list)

# get 1000 most variable genes
sc.pp.highly_variable_genes(adata, n_top_genes=1000)

# only keep variable genes and genes from paper
adata = adata[:,adata.var['highly_variable'] | adata.var['gene_list']].copy()